# Clase 4 · Optimización I: cómo aprende una red

**Curso: Advanced Deep Learning**

Este cuaderno es el que ejecutas en vivo durante la clase. Combina explicaciones breves, tipo apuntes, con celdas de código que corres y modificas en el momento. El objetivo no es memorizar fórmulas, sino construir intuición sobre el algoritmo que está detrás de todo entrenamiento de una red: el descenso de gradiente (*gradient descent*, GD).

## Objetivos de aprendizaje

Al terminar esta clase, sabrás:

- Explicar qué es una función objetivo (o función de pérdida) y por qué entrenar una red equivale a minimizarla
- Implementar el descenso de gradiente a mano, sin ningún framework, usando solo NumPy
- Explicar cómo determina la tasa de aprendizaje si el entrenamiento converge, diverge u oscila
- Reconocer el número de condición de una superficie de pérdida y explicar por qué un valor alto complica el entrenamiento
- Conectar la teoría (constante de Lipschitz, tasa máxima estable $1/L$) con lo que observas en tus propias gráficas


## 1. La función objetivo como un paisaje

Imagina que estás en la cima de una montaña, envuelto en niebla espesa. Solo ves el terreno bajo tus pies. Tu objetivo es llegar al punto más bajo del valle, y no tienes mapa: en cada punto sientes hacia dónde baja más el terreno y das un paso en esa dirección.

Esa es la idea del descenso de gradiente. La "montaña" es la función objetivo $h(\theta)$: mide cuánto se equivoca tu modelo para cada combinación de parámetros $\theta$. Tu "posición" son los valores actuales de esos parámetros. El "sentido de la pendiente" es el gradiente $\nabla h(\theta)$, un vector que apunta hacia donde la función crece más rápido. Como quieres bajar, te mueves en la dirección contraria: el gradiente negativo.

Dibuja primero ese paisaje con una función de pérdida cuadrática simple de dos parámetros, $\theta_1$ y $\theta_2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# h(theta) = 0.5 * (a * theta_1^2 + b * theta_2^2)
# 'a' y 'b' son las curvaturas en cada dirección del paisaje
a, b = 1.0, 5.0

def loss(theta):
    return 0.5 * (a * theta[0] ** 2 + b * theta[1] ** 2)

theta1 = np.linspace(-6, 6, 200)
theta2 = np.linspace(-6, 6, 200)
T1, T2 = np.meshgrid(theta1, theta2)
Z = 0.5 * (a * T1 ** 2 + b * T2 ** 2)

fig = plt.figure(figsize=(13, 5))

# Panel izquierdo: curvas de nivel, la vista "desde arriba" de la montaña
ax1 = fig.add_subplot(1, 2, 1)
ax1.contour(T1, T2, Z, levels=20, cmap="viridis")
ax1.plot(0, 0, "r*", markersize=15, label="Mínimo")
ax1.set_xlabel("theta_1")
ax1.set_ylabel("theta_2")
ax1.set_title("Curvas de nivel de h(theta)")
ax1.legend()

# Panel derecho: la superficie 3D, la "montaña" real de la analogía
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.plot_surface(T1, T2, Z, cmap="viridis", alpha=0.85)
ax2.set_xlabel("theta_1")
ax2.set_ylabel("theta_2")
ax2.set_zlabel("h(theta)")
ax2.set_title("La función objetivo como paisaje")

plt.tight_layout()
plt.show()

## 2. Descenso de gradiente paso a paso

La regla de actualización del descenso de gradiente es:

$$\theta_{k+1} = \theta_k - \alpha \, \nabla h(\theta_k)$$

En cada iteración $k$, calculas el gradiente en tu posición actual y das un paso de tamaño $\alpha$ (la tasa de aprendizaje) en la dirección contraria. Repites el proceso hasta acercarte al mínimo.

Implementa esta regla a mano, sin ningún framework, y guarda la trayectoria completa de $\theta$ para poder dibujarla sobre el paisaje que acabas de ver.

In [ ]:
def gradient(theta):
    # Derivadas parciales de la cuadrática respecto a cada parámetro
    return np.array([a * theta[0], b * theta[1]])

def gradient_descent(theta_init, alpha, num_steps):
    theta = np.array(theta_init, dtype=float)
    trayectoria = [theta.copy()]
    for _ in range(num_steps):
        theta = theta - alpha * gradient(theta)  # regla de actualización de GD
        trayectoria.append(theta.copy())
        if np.any(np.abs(theta) > 1e6):  # evita desbordar el tipo float si diverge
            break
    return np.array(trayectoria)

theta_inicial = [4.0, 4.0]
trayectoria = gradient_descent(theta_inicial, alpha=0.15, num_steps=25)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.contour(T1, T2, Z, levels=20, cmap="viridis", alpha=0.6)
ax.plot(trayectoria[:, 0], trayectoria[:, 1], "o-", color="crimson", markersize=4, label="Trayectoria de GD")
ax.plot(0, 0, "r*", markersize=15, label="Mínimo")
ax.set_xlabel("theta_1")
ax.set_ylabel("theta_2")
ax.set_title("Descenso de gradiente sobre el paisaje")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Punto de partida: {theta_inicial}")
print(f"Punto final tras {len(trayectoria) - 1} pasos: {trayectoria[-1]}")
print(f"Pérdida inicial: {loss(trayectoria[0]):.4f} -> Pérdida final: {loss(trayectoria[-1]):.6f}")

## 3. El papel crítico de la tasa de aprendizaje

La tasa de aprendizaje decide el tamaño del paso, y de ella depende todo:

- **Paso adecuado**: avanzas de forma constante y razonablemente rápida hacia el mínimo
- **Paso demasiado grande**: saltas por encima del valle y aterrizas más arriba de donde empezaste; repite esto varias veces y el proceso diverge
- **Paso demasiado pequeño**: avanzas con total seguridad, pero tan despacio que el entrenamiento se vuelve impracticable

La teoría te da una cota: con $L$ como la curvatura máxima de la función (la constante de Lipschitz), una tasa $\alpha \le 1/L$ garantiza que cada paso no aumente la pérdida. Compara ahora tres tasas distintas sobre el mismo paisaje.

In [ ]:
num_pasos = 40
L = max(a, b)
alpha_max = 1.0 / L  # tasa máxima teórica de estabilidad conservadora

tasas = {
    "Buena (alpha = 0.15)": 0.15,
    "Demasiado alta (alpha = 0.45)": 0.45,   # supera 2/L y el proceso diverge
    "Demasiado baja (alpha = 0.02)": 0.02,   # converge, pero muy despacio
}

resultados = {
    nombre: gradient_descent(theta_inicial, alpha, num_pasos)
    for nombre, alpha in tasas.items()
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: las tres trayectorias sobre el contour plot
ax = axes[0]
ax.contour(T1, T2, Z, levels=25, cmap="viridis", alpha=0.6)
for nombre, trayectoria in resultados.items():
    ax.plot(trayectoria[:, 0], trayectoria[:, 1], marker="o", markersize=3, label=nombre)
ax.plot(0, 0, "r*", markersize=15, label="Mínimo")
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_xlabel("theta_1")
ax.set_ylabel("theta_2")
ax.set_title("Trayectorias según la tasa de aprendizaje")
ax.legend(fontsize=8)

# Panel derecho: evolución de la pérdida en escala logarítmica
ax = axes[1]
for nombre, trayectoria in resultados.items():
    valores_perdida = [loss(theta) for theta in trayectoria]
    ax.plot(valores_perdida, marker="o", markersize=3, label=nombre)
ax.set_yscale("log")
ax.set_xlabel("Iteración")
ax.set_ylabel("Pérdida h(theta) [escala log]")
ax.set_title("Convergencia según la tasa de aprendizaje")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Tasa de aprendizaje máxima estable (1/L) = {alpha_max:.4f}")

## 4. Curvatura y número de condición

Ahora complica el paisaje: en vez de un valle en forma de cuenco, imagina un cañón estrecho y alargado, con paredes empinadas a los lados y un fondo casi plano. Si das pasos grandes, rebotas de pared a pared sin apenas avanzar por el fondo: oscilas en lugar de progresar. Si reduces el paso para evitar el rebote, avanzas seguro por las paredes, pero te arrastras a lo largo del fondo plano. No existe una tasa única que resuelva bien las dos direcciones a la vez.

La relación entre la curvatura máxima y la mínima se llama número de condición, $\kappa = L/\mu$. Cuanto mayor sea $\kappa$, más estrecho es el cañón y más iteraciones necesitas: el número de pasos crece de forma proporcional a $\kappa$. Sube ahora el número de condición y observa el zigzag.

In [ ]:
a2, b2 = 1.0, 100.0  # numero de condicion kappa = b2 / a2 = 100: canon mucho mas estrecho
L2 = max(a2, b2)

def gradient2(theta):
    return np.array([a2 * theta[0], b2 * theta[1]])

def loss2(theta):
    return 0.5 * (a2 * theta[0] ** 2 + b2 * theta[1] ** 2)

def gradient_descent2(theta_init, alpha, num_steps):
    theta = np.array(theta_init, dtype=float)
    trayectoria = [theta.copy()]
    for _ in range(num_steps):
        theta = theta - alpha * gradient2(theta)
        trayectoria.append(theta.copy())
    return np.array(trayectoria)

theta_inicial2 = [4.0, 4.0]
alpha2 = 0.018  # una tasa "razonable" para la direccion plana, pero cercana al limite en la empinada
trayectoria2 = gradient_descent2(theta_inicial2, alpha2, num_steps=60)

theta1_r = np.linspace(-6, 6, 200)
theta2_r = np.linspace(-6, 6, 200)
T1b, T2b = np.meshgrid(theta1_r, theta2_r)
Zb = 0.5 * (a2 * T1b ** 2 + b2 * T2b ** 2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.contour(T1b, T2b, Zb, levels=30, cmap="viridis", alpha=0.6)
ax.plot(trayectoria2[:, 0], trayectoria2[:, 1], "o-", markersize=3, color="crimson")
ax.plot(0, 0, "r*", markersize=15, label="Mínimo")
ax.set_xlabel("theta_1 (curvatura baja)")
ax.set_ylabel("theta_2 (curvatura alta)")
ax.set_title(f"Zigzag con numero de condicion kappa = {b2 / a2:.0f}")
ax.legend()

ax = axes[1]
valores2 = [loss2(theta) for theta in trayectoria2]
ax.plot(valores2, "o-", markersize=3, color="crimson")
ax.set_yscale("log")
ax.set_xlabel("Iteración")
ax.set_ylabel("Pérdida h(theta) [escala log]")
ax.set_title("La pérdida baja, pero con muchas oscilaciones")

plt.tight_layout()
plt.show()

print(f"Numero de condicion (kappa) = {b2 / a2:.0f}")
print(f"Tasa de aprendizaje maxima estable (1/L) = {1 / L2:.4f}")
print(f"Perdida tras {len(trayectoria2) - 1} pasos: {loss2(trayectoria2[-1]):.4f} (compara con la seccion anterior)"
)

## Resumen de puntos clave

- Entrenar una red equivale a minimizar una función objetivo $h(\theta)$ moviendo sus parámetros paso a paso
- El descenso de gradiente sigue la regla $\theta_{k+1} = \theta_k - \alpha \nabla h(\theta_k)$: te mueves en dirección contraria al gradiente, la dirección de mayor crecimiento de la pérdida
- La tasa de aprendizaje $\alpha$ controla el tamaño del paso: demasiado alta y el proceso oscila o diverge; demasiado baja y converge, pero muy despacio
- La constante de Lipschitz $L$ acota la curvatura máxima; una tasa $\alpha \le 1/L$ garantiza estabilidad conservadora
- El número de condición $\kappa = L/\mu$ mide cuánto más empinada es la dirección más curvada frente a la más plana; cuanto mayor sea $\kappa$, más estrecho es el "cañón" y más pasos necesitas para converger
- Un único valor de $\alpha$ nunca resuelve bien un cañón muy estrecho: lo que estabiliza la dirección empinada resulta lentísimo en la dirección plana

**Abre ahora el cuaderno de práctica companion, `../notebooks_practica/clase04_practica_optimizacion_fundamentos.ipynb`, y pon a prueba tú mismo estas ideas.**